In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
sentences = [
    "machine learning engineer",
    "специалист по искусственному интеллекту",
    "chef in a restaurant",
]
embeddings = model.encode(sentences)
embeddings.shape

(3, 384)

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import sys
sys.path.append("..")
from src.text_cleaning import clean_text
scores = cosine_similarity(embeddings)
pd.DataFrame(scores, index=sentences, columns=sentences).round(2)

,machine learning engineer,специалист по искусственному интеллекту,chef in a restaurant
machine learning engineer,1.00,0.66,0.23
специалист по искусственному интеллекту,0.66,1.00,0.19
chef in a restaurant,0.23,0.19,1.00


In [4]:
df = pd.read_csv('../data/eval/Resume.csv')
df['clean_text'] = df['Resume_str'].apply(clean_text)


In [5]:
import os
import numpy as np

EMB_PATH = "../data/eval/resume_embeddings.npy"

if os.path.exists(EMB_PATH):
    # файл уже есть: просто загружаем, это быстро
    resume_embeddings = np.load(EMB_PATH)
else:
    # файла нет: считаем один раз и сохраняем
    resume_embeddings = model.encode(df['clean_text'].tolist(), show_progress_bar=True)
    np.save(EMB_PATH, resume_embeddings)

resume_embeddings.shape

(2484, 384)

In [6]:
def split_into_chunks(text, chunk_size=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk_words = words[i : i + chunk_size]
        chunks.append(" ".join(chunk_words))
    return chunks

split_into_chunks("один два три четыре пять шесть семь", chunk_size=3)

['один два три', 'четыре пять шесть', 'семь']

In [7]:
def embed_long_text(text):
    chunks = split_into_chunks(text)
    if len(chunks) == 0:
        return np.zeros(384)
    chunk_vecs = model.encode(chunks)
    return chunk_vecs.mean(axis=0)      

vec = embed_long_text(df['clean_text'][0])
vec.shape

(384,)

In [8]:
from tqdm import tqdm

CHUNK_EMB_PATH = "../data/eval/resume_embeddings_chunked.npy"

if os.path.exists(CHUNK_EMB_PATH):
    resume_embeddings_chunked = np.load(CHUNK_EMB_PATH)
else:
    vecs = [embed_long_text(t) for t in tqdm(df['clean_text'])]
    resume_embeddings_chunked = np.array(vecs)
    np.save(CHUNK_EMB_PATH, resume_embeddings_chunked)

resume_embeddings_chunked.shape

(2484, 384)

In [ ]:
def search(vacancy_text, k=5):
    vacancy_clean = clean_text(vacancy_text)
    vacancy_vec = model.encode([vacancy_clean])
    scores = cosine_similarity(vacancy_vec, resume_embeddings)[0]
    result = df.copy()
    result['score'] = scores
    return result.sort_values('score', ascending=False).head(k)

search("experienced accountant with tax and bookkeeping skills")[['Category', 'score']]

,Category,score
1823,ACCOUNTANT,0.692595
1876,ACCOUNTANT,0.668104
1833,ACCOUNTANT,0.648678
1820,ACCOUNTANT,0.629025
1847,ACCOUNTANT,0.620047


Усреднение по кускам ухудшило результат, потому что размывает главную часть резюме (должность и summary). Оставляем вариант с началом резюме

In [10]:
vacancies = {
    "ACCOUNTANT": "Accountant to manage general ledger, accounts payable and receivable, tax returns, reconciliations and monthly financial reporting.",
    "ADVOCATE": "Attorney to provide legal advice, draft contracts, conduct legal research and represent clients in court and litigation.",
    "AGRICULTURE": "Agriculture specialist to manage crop production, farm operations, soil and irrigation, livestock and agronomy programs.",
    "APPAREL": "Apparel merchandiser to manage clothing collections, garment production, fashion trends, fabric sourcing and retail buying.",
    "ARTS": "Arts program coordinator to organize exhibitions, theater and music events, work with artists and manage art education programs.",
    "AUTOMOBILE": "Automotive technician to diagnose and repair vehicles, engines, brakes and transmissions, and perform vehicle maintenance.",
    "AVIATION": "Aviation maintenance technician to inspect and repair aircraft, follow FAA regulations and support flight operations.",
    "BANKING": "Banking officer to open customer accounts, process loans and deposits, ensure compliance and provide branch customer service.",
    "BPO": "BPO call center agent to handle inbound and outbound customer calls, resolve client issues and meet process quality targets.",
    "BUSINESS-DEVELOPMENT": "Business development manager to find new clients, build partnerships, generate leads and grow company revenue.",
    "CHEF": "Chef to prepare dishes, plan menus, manage the kitchen staff, ensure food safety and control food costs.",
    "CONSTRUCTION": "Construction project manager to supervise building sites, contractors, safety, schedules and construction budgets.",
    "CONSULTANT": "Consultant to analyze client business problems, recommend solutions, lead projects and present findings to stakeholders.",
    "DESIGNER": "Designer to create visual concepts, layouts and branding using Adobe Photoshop, Illustrator and InDesign.",
    "DIGITAL-MEDIA": "Digital media specialist to manage social media, create video and web content, run online campaigns and analyze engagement.",
    "ENGINEERING": "Engineer to design mechanical and electrical systems, use AutoCAD, perform testing and manage engineering projects.",
    "FINANCE": "Financial analyst to build financial models, forecasts and budgets, analyze investments and prepare financial statements.",
    "FITNESS": "Fitness trainer to lead personal training sessions and group exercise classes, design workout programs and coach clients.",
    "HEALTHCARE": "Healthcare professional to provide patient care, maintain medical records, assist physicians and follow clinical procedures.",
    "HR": "HR specialist to manage recruitment, onboarding, employee relations, payroll, benefits and HR policies.",
    "INFORMATION-TECHNOLOGY": "IT specialist to support networks, servers and help desk, manage systems administration and troubleshoot hardware and software.",
    "PUBLIC-RELATIONS": "Public relations specialist to write press releases, manage media relations, plan events and handle corporate communications.",
    "SALES": "Sales representative to sell products, meet sales targets, manage customer accounts and negotiate deals.",
    "TEACHER": "Teacher to plan lessons, teach students in the classroom, assess student progress and communicate with parents.",
}


results = []

for category, text in vacancies.items():
    top = search(text)
    precision = (top['Category'] == category).mean()
    results.append({"category": category, "precision@5": precision})

results_df = pd.DataFrame(results)
results_df

print(results_df['precision@5'].mean())

0.725


In [11]:
model.max_seq_length


128

In [12]:
df['clean_text'].str.split().str.len().mean()

np.float64(792.3289049919484)

In [13]:
chunks = split_into_chunks(df['clean_text'][0])
len(chunks)

7